In [8]:
import pandas as pd
import numpy as np
import json
import numpy as np

In [1]:
from pathlib import Path
import pandas as pd

# Define the path
file_path = Path(
    "/home/divas/ml/CI-diagnosis-agent/data/benchmark_data/final_test_cases.jsonl"
)

# Load into a DataFrame
df = pd.read_json(file_path, lines=True)

# Preview the data
df.head()

,test_id,split,scenario_description,ground_truth,evidence,scenario_optimal_action,generation_metadata,expected_agent_action,hard_case_tags
0,easy_000001,medium,The event-bus CI pipeline failed during the De...,"{'state': 'S4', 'failure_mechanism': 'import_o...","{'E1': {'outcome': 'A', 'observation': 'The de...","{'optimal_action': 'fix_code', 'ground_action'...","{'generation_method': 'scenario_renderer', 'di...",Fix lint,NaN
1,easy_000002,easy,The analytics-platform repository's Continuous...,"{'state': 'S1', 'failure_mechanism': 'logic_bug'}","{'E1': {'outcome': 'C', 'observation': 'The te...","{'optimal_action': 'fix_code', 'ground_action'...","{'generation_method': 'scenario_renderer', 'di...",Escalate,NaN
2,easy_000003,easy,A pull request in the ml-model-serving project...,"{'state': 'S3', 'failure_mechanism': 'dependen...","{'E1': {'outcome': 'A', 'observation': 'The de...","{'optimal_action': 'resolve_dependency', 'grou...","{'generation_method': 'scenario_renderer', 'di...",Fix dependency,NaN
3,easy_000004,easy,The api-gateway repository's Smoke Tests workf...,"{'state': 'S4', 'failure_mechanism': 'lint_vio...","{'E1': {'outcome': 'D', 'observation': 'The st...","{'optimal_action': 'fix_code', 'ground_action'...","{'generation_method': 'scenario_renderer', 'di...",Fix lint,NaN
4,easy_000005,easy,The user-service CI pipeline failed during the...,"{'state': 'S5', 'failure_mechanism': 'mock_fai...","{'E1': {'outcome': 'C', 'observation': 'The te...","{'optimal_action': 'investigate_tests', 'groun...","{'generation_method': 'scenario_renderer', 'di...",Escalate,NaN


In [2]:
import pandas as pd

# Assuming your original DataFrame is named 'df'


# Function to process each row
def process_row(row):
    gt = row.get("ground_truth") or {}
    ev = row.get("evidence") or {}
    soa = row.get("scenario_optimal_action") or {}

    processed = {
        "test_id": row.get("test_id"),
        "split": row.get("split"),
        "scenario_description": row.get("scenario_description"),
        "ground_truth": (
            gt.get("state") if isinstance(gt, dict) else None
        ),  # Extracts state (e.g. S4, S1)
        # Evidence outcomes
        "E1_outcome": (
            ev.get("E1", {}).get("outcome") if isinstance(ev, dict) else None
        ),
        "E2_outcome": (
            ev.get("E2", {}).get("outcome") if isinstance(ev, dict) else None
        ),
        "E3_outcome": (
            ev.get("E3", {}).get("outcome") if isinstance(ev, dict) else None
        ),
        "E4_outcome": (
            ev.get("E4", {}).get("outcome") if isinstance(ev, dict) else None
        ),
        # Evidence observations
        "E1_observation": (
            ev.get("E1", {}).get("observation") if isinstance(ev, dict) else None
        ),
        "E2_observation": (
            ev.get("E2", {}).get("observation") if isinstance(ev, dict) else None
        ),
        "E3_observation": (
            ev.get("E3", {}).get("observation") if isinstance(ev, dict) else None
        ),
        "E4_observation": (
            ev.get("E4", {}).get("observation") if isinstance(ev, dict) else None
        ),
        # Optimal scenario action fields
        "optimal_scenario_action": (
            soa.get("optimal_action") if isinstance(soa, dict) else None
        ),
        "ground_action": (
            soa.get("ground_action") if isinstance(soa, dict) else None
        ),
        "description": (
            soa.get("description") if isinstance(soa, dict) else None
        ),
    }

    return pd.Series(processed)


# Apply the processing across all rows
df = df.apply(process_row, axis=1)

# View the formatted DataFrame
df.head()

,test_id,split,scenario_description,ground_truth,E1_outcome,E2_outcome,E3_outcome,E4_outcome,E1_observation,E2_observation,E3_observation,E4_observation,optimal_scenario_action,ground_action,description
0,easy_000001,medium,The event-bus CI pipeline failed during the De...,S4,A,src,pass_on_rerun,not_reproducible_locally,The dependency installation step was the first...,The eventual successful repair modified applic...,Rerunning the exact same commit on CI resulted...,The failure could not be reproduced in a local...,fix_code,Fix Lint,The failure is a static-analysis violation tha...
1,easy_000002,easy,The analytics-platform repository's Continuous...,S1,C,src,fail_on_rerun,reproducible_locally,The test execution step was the first failed s...,The eventual successful repair modified applic...,Rerunning the exact same commit on CI produced...,The developer reproduced the same failure in t...,fix_code,Escalate,The failure originates from a defect in the ap...
2,easy_000003,easy,A pull request in the ml-model-serving project...,S3,A,config,fail_on_rerun,not_reproducible_locally,The dependency installation step was the first...,The eventual successful repair modified projec...,Rerunning the exact same commit on CI produced...,The failure could not be reproduced in a local...,resolve_dependency,Fix Dependency,The failure is caused by a dependency manageme...
3,easy_000004,easy,The api-gateway repository's Smoke Tests workf...,S4,D,src,fail_on_rerun,reproducible_locally,The static analysis or quality checks step was...,The eventual successful repair modified applic...,Rerunning the exact same commit on CI produced...,The developer reproduced the same failure in t...,fix_code,Fix Lint,The failure is a static-analysis violation tha...
4,easy_000005,easy,The user-service CI pipeline failed during the...,S5,C,test,fail_on_rerun,reproducible_locally,The test execution step was the first failed s...,The eventual successful repair modified test f...,Rerunning the exact same commit on CI produced...,The developer reproduced the same failure in t...,investigate_tests,Escalate,The failure is a test execution problem that r...


### 1.Data Analysis

In [5]:
df['ground_truth'].value_counts(normalize = True)

ground_truth
S3    0.222222
S5    0.189542
S4    0.183007
S1    0.143791
S7    0.104575
S6    0.084967
S2    0.071895
Name: proportion, dtype: float64

In [6]:
df['ground_action'].value_counts(normalize = True)

ground_action
Escalate          0.500000
Fix Dependency    0.296053
Fix Lint          0.203947
Name: proportion, dtype: float64

In [12]:
df.shape

(153, 15)

### 1.Dump Policy - Always escelate



In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [49]:

# Find the most common ground-truth action
p0_action = df["ground_action"].mode()[0]
print("P0 baseline action:", p0_action)

P0 baseline action: Escalate


In [50]:
df["p0_prediction"] = p0_action

In [51]:
# ============================================================
# 2. ACCURACY
# ============================================================

accuracy = accuracy_score(
    df["ground_action"],
    df["p0_prediction"]
)

print(f"\nAccuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

# ============================================================
# 3. PRECISION / RECALL / F1
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        df["ground_action"],
        df["p0_prediction"],
        labels=sorted(df["ground_action"].unique()),
        zero_division=0
    )
)

# Macro metrics
precision_macro = precision_score(
    df["ground_action"],
    df["p0_prediction"],
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    df["ground_action"],
    df["p0_prediction"],
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    df["ground_action"],
    df["p0_prediction"],
    average="macro",
    zero_division=0
)

# Weighted metrics
precision_weighted = precision_score(
    df["ground_action"],
    df["p0_prediction"],
    average="weighted",
    zero_division=0
)

recall_weighted = recall_score(
    df["ground_action"],
    df["p0_prediction"],
    average="weighted",
    zero_division=0
)

f1_weighted = f1_score(
    df["ground_action"],
    df["p0_prediction"],
    average="weighted",
    zero_division=0
)

print("\nAggregate Metrics")
print("-----------------")
print(f"Macro Precision   : {precision_macro:.4f}")
print(f"Macro Recall      : {recall_macro:.4f}")
print(f"Macro F1          : {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall   : {recall_weighted:.4f}")
print(f"Weighted F1       : {f1_weighted:.4f}")

# ============================================================
# 4. CONFUSION MATRIX
# ============================================================

labels = sorted(df["ground_action"].unique())

cm = confusion_matrix(
    df["ground_action"],
    df["p0_prediction"],
    labels=labels
)

print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=labels, columns=labels))

# Plot
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("P0 Baseline Confusion Matrix")
plt.tight_layout()
plt.show()

TypeError: '<' not supported between instances of 'float' and 'str'